# paths

> Path utilities and project discovery for nbdev projects

In [ ]:
#| default_exp utils.paths

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional, Iterable, Tuple
from functools import lru_cache

import os
import re
import json
import time
import tomllib
import platform
from pathlib import Path
from configparser import ConfigParser

from nbdev_mcp.utils.config import (
    CURRENT_PROJECT,
    load_bookmarks,
    EXPECTED_PROMPT_TEMPLATES
)
from nbdev_mcp.utils.nb import find_default_exp

from nbdev_mcp.utils.re import (
    DEFAULT_EXP_PATTERN,
    YAML_NAME_PATTERN,
    RELATIVE_LEVEL_PATTERN,
    DIGIT_PREFIX,
    NBDEV_FILE_PREFIX
)


## Path Expansion

In [ ]:
#| export
def expand(p: str) -> Path:
    """Expand user/home variables and return an absolute Path.
    
    Parameters
    ----------
    p : str
        Path string, may contain ``~`` or environment variables.
    
    Returns
    -------
    Path
        Resolved absolute path.
    
    Examples
    --------
    >>> expand('~/projects')
    PosixPath('/Users/username/projects')
    """
    return Path(os.path.expanduser(os.path.expandvars(p))).resolve()

## Settings

In [ ]:
#| export
def settings_dict(project: Path) -> Dict[str, str]:
    """Read nbdev settings from .ini/.toml config files.
    
    Detection order:
    1. ``settings.toml``
    2. ``pyproject.toml`` under ``[tool.nbdev]``
    3. ``settings.ini``
    
    Parameters
    ----------
    project : Path
        Path to the nbdev project root.
    
    Returns
    -------
    Dict[str, str]
        Dictionary with normalized string values. Missing keys are empty strings.
    """
    fields = [
        "lib_name",
        "lib_path",
        "nbs_path",
        "doc_path",
        "branch",
        "repo",
        "user",
        "black_formatting",
        "console_scripts",
        "version",
        "description",
        "author",
        "requirements",
        "dev_requirements",
        "nbdev_version",
    ]

    settings_toml = project / "settings.toml"
    pyproject_toml = project / "pyproject.toml"

    toml_candidates: List[Dict[str, Any]] = []
    for toml_path in [settings_toml, pyproject_toml]:
        if not toml_path.exists():
            continue
        try:
            toml_data = tomllib.loads(toml_path.read_text(encoding="utf-8"))
        except Exception:
            continue

        if isinstance(toml_data, dict):
            tool = toml_data.get("tool")
            if isinstance(tool, dict):
                nbdev_cfg = tool.get("nbdev")
                if isinstance(nbdev_cfg, dict):
                    toml_candidates.append(nbdev_cfg)

            nbdev_top = toml_data.get("nbdev")
            if isinstance(nbdev_top, dict):
                toml_candidates.append(nbdev_top)

            toml_candidates.append(toml_data)

    for candidate in toml_candidates:
        if not any(key in candidate for key in fields):
            continue
        parsed: Dict[str, str] = {}
        for key in fields:
            value = candidate.get(key, "")
            if isinstance(value, list):
                parsed[key] = " ".join(str(item).strip() for item in value if str(item).strip())
            elif value is None:
                parsed[key] = ""
            else:
                parsed[key] = str(value).strip()
        return parsed

    cfg = ConfigParser()
    settings_ini = project / "settings.ini"
    if not settings_ini.exists():
        return {}

    cfg.read(settings_ini)
    data = dict(cfg["DEFAULT"]) if "DEFAULT" in cfg else {}

    parsed_ini: Dict[str, str] = {}
    for key in fields:
        parsed_ini[key] = data.get(key, "").strip()
    return parsed_ini


In [ ]:
#| export
def nbdev_settings_path(project: Path) -> Optional[Path]:
    """Return the active nbdev settings file for a project.
    
    Preference order:
    1. ``settings.toml``
    2. ``pyproject.toml`` with ``[tool.nbdev]``
    3. ``settings.ini``
    """
    settings_toml = project / "settings.toml"
    if settings_toml.exists():
        return settings_toml

    pyproject_toml = project / "pyproject.toml"
    if pyproject_toml.exists():
        try:
            pyproject_data = tomllib.loads(pyproject_toml.read_text(encoding="utf-8"))
            tool = pyproject_data.get("tool") if isinstance(pyproject_data, dict) else None
            if isinstance(tool, dict) and isinstance(tool.get("nbdev"), dict):
                return pyproject_toml
        except Exception:
            pass

    settings_ini = project / "settings.ini"
    if settings_ini.exists():
        return settings_ini

    return None


In [ ]:
#| export
def nbdev_generation(project: Path) -> str:
    """Detect nbdev generation for a project (v2/v3/unknown)."""
    settings_file = nbdev_settings_path(project)
    if settings_file is None:
        return "unknown"

    if settings_file.name == "settings.ini":
        return "v2"

    settings = settings_dict(project)
    nbdev_version = settings.get("nbdev_version", "")
    if nbdev_version:
        major_digits = ""
        for ch in nbdev_version.strip():
            if ch.isdigit():
                major_digits += ch
            elif major_digits:
                break
        if major_digits:
            try:
                return "v3" if int(major_digits) >= 3 else "v2"
            except ValueError:
                pass

    if settings_file.suffix == ".toml":
        return "v3"

    return "unknown"


In [ ]:
#| export
def nbdev_command_name(project: Path, command: str) -> str:
    """Return the nbdev CLI command name for the detected project generation.
    
    Parameters
    ----------
    project : Path
        Path to the nbdev project root.
    command : str
        Command key or command name.
        Accepted keys: ``prepare``, ``export``, ``test``, ``readme``.
    """
    normalized = command.strip()
    if normalized.startswith("nbdev_"):
        normalized = normalized[6:]
    elif normalized.startswith("nbdev-"):
        normalized = normalized[6:]

    command_table = {
        "prepare": ("nbdev_prepare", "nbdev-prepare"),
        "export": ("nbdev_export", "nbdev-export"),
        "test": ("nbdev_test", "nbdev-test"),
        "readme": ("nbdev_readme", "nbdev-readme"),
    }

    names = command_table.get(normalized)
    if names is None:
        return command

    generation = nbdev_generation(project)
    if generation == "v3":
        return names[1]
    return names[0]


In [ ]:
#| export
def lib_name(project: Path) -> str:
    """Get library name from nbdev settings, or 'pkg' as fallback.
    
    Parameters
    ----------
    project : Path
        Path to the nbdev project root.
    
    Returns
    -------
    str
        Library name from nbdev config or 'pkg' if not found.
    """
    return settings_dict(project).get('lib_name') or 'pkg'


## Project Directories

In [ ]:
#| export
def nbs_dir(project: Path) -> Path:
    """Get the notebook directory Path for the project.
    
    Parameters
    ----------
    project : Path
        Path to the nbdev project root.
    
    Returns
    -------
    Path
        Resolved path to notebooks directory (from nbs_path setting or 'nbs').
    """
    s = settings_dict(project)
    return (project / (s.get("nbs_path") or "nbs")).resolve()

In [ ]:
#| export
def tutorials_dir(project: Path) -> Path:
    """Return the tutorials directory path for the project.
    
    Parameters
    ----------
    project : Path
        Path to the nbdev project root.
    
    Returns
    -------
    Path
        Resolved path to ``{project}/tutorials``.
    """
    return (project / "tutorials").resolve()

In [ ]:
#| export
def safe_relative_to(path: Path, base: Path, fallback_base: Optional[Path] = None) -> str:
    """Safely compute relative path, with fallback for paths outside base.
    
    Parameters
    ----------
    path : Path
        The path to make relative.
    base : Path
        The primary base directory to make path relative to.
    fallback_base : Path, optional
        If path is not under base, try this as fallback.
        If None and path is not under base, returns path.name.
    
    Returns
    -------
    str
        Relative path string, or just the filename if path is outside all bases.
    
    Examples
    --------
    >>> safe_relative_to(Path('/proj/nbs/foo.ipynb'), Path('/proj/nbs'))
    'foo.ipynb'
    >>> safe_relative_to(Path('/proj/tutorials/bar.ipynb'), Path('/proj/nbs'), Path('/proj'))
    'tutorials/bar.ipynb'
    """
    try:
        return str(path.relative_to(base))
    except ValueError:
        if fallback_base is not None:
            try:
                return str(path.relative_to(fallback_base))
            except ValueError:
                pass
        return path.name

In [ ]:
#| export
def is_nbdev_project(p: Path) -> bool:
    """Check if the given Path is an nbdev project.
    
    Parameters
    ----------
    p : Path
        Path to check.
    
    Returns
    -------
    bool
        True if path has a recognized nbdev settings file and nbs directory.
    """
    settings_file = nbdev_settings_path(p)
    return (settings_file is not None) and nbs_dir(p).exists()


## Environment Files

In [ ]:
#| export
def env_file(
    project: Path | None = None,
    scoped: bool = True,
    envs_dir: Path | None = None,
    lib_name_override: str | None = None,
) -> Path:
    """Resolve the environment YAML path with sensible fallbacks.
    
    Searches for environment files in this priority order:
    
    1. ``{envs_dir}/env.{lib_name}.yml`` (scoped, custom dir)
    2. ``{envs_dir}/env.yml`` (unscoped, custom dir)
    3. ``{project}/env.{lib_name}.yml`` (scoped, project root)
    4. ``{project}/env.yml`` (unscoped, project root)
    5. ``{project}/../Environments/env.{lib_name}.yml`` (sibling dir)
    6. ``{project}/../Environments/env.yml`` (sibling dir)
    
    Parameters
    ----------
    project : Path or None
        The nbdev project root. Defaults to current directory.
    scoped : bool
        If True, prefer ``env.{lib_name}.yml`` over ``env.yml``.
    envs_dir : Path or None
        Custom directory to search first for environment files.
    lib_name_override : str or None
        Library name for scoped files. Defaults to lib_name from nbdev settings.
    
    Returns
    -------
    Path
        The first existing environment file, or the first candidate if none exist.
    """
    root = (project or Path()).resolve()
    lib = lib_name_override or settings_dict(root).get("lib_name", "env")
    env_roots = [envs_dir, root, root.parent / "Environments"]
    candidates: list[Path] = []
    for base in env_roots:
        if base is None:
            continue
        base = Path(base)
        if scoped:
            candidates.append(base / f"env.{lib}.yml")
        candidates.append(base / "env.yml")
    for cand in candidates:
        if cand.exists():
            return cand.resolve()
    return candidates[0].resolve()


In [ ]:
#| export
def discover_env_name(project: Path) -> Optional[str]:
    """Read the env YAML file and extract the environment 'name' field.
    
    Parameters
    ----------
    project : Path
        Path to the nbdev project root.
    
    Returns
    -------
    str or None
        Environment name if found, None otherwise.
    """
    yml = env_file(project)
    if not yml.exists():
        return None
    
    for line in yml.read_text(encoding="utf-8").splitlines():
        m = YAML_NAME_PATTERN.match(line)
        if m:
            return m.group(1)
    return None

## Project Discovery

In [ ]:
#| export
def find_project_root(start: Path) -> Optional[Path]:
    """Ascend from the given path to find the root of an nbdev project.
    
    Parameters
    ----------
    start : Path
        Starting path to search from.
    
    Returns
    -------
    Path or None
        Project root path, or None if not found.
    """
    p = start.resolve()
    if p.is_file():
        p = p.parent
    while True:
        if is_nbdev_project(p):
            return p
        if p.parent == p:
            return None  # reached filesystem root
        p = p.parent

In [ ]:
#| export
def require_project() -> Path:
    """Get the currently active project Path, defaulting to cwd if valid.
    
    Resolution order:
    1. If CURRENT_PROJECT is explicitly set, return it
    2. If current working directory is an nbdev project, use it
    3. If cwd is inside an nbdev project, use the project root
    4. Otherwise raise RuntimeError
    
    This enables single-project mode (just cd into your project) while
    still supporting multi-project workflows via set_project().
    
    Returns
    -------
    Path
        The resolved project path.
    
    Raises
    ------
    RuntimeError
        If no project is selected and cwd is not an nbdev project.
    """
    if CURRENT_PROJECT is not None:
        return CURRENT_PROJECT
    
    # Default to cwd if it's an nbdev project
    cwd = Path.cwd()
    if is_nbdev_project(cwd):
        return cwd
    
    # Check if we're in a subdirectory of an nbdev project
    root = find_project_root(cwd)
    if root is not None:
        return root
    
    raise RuntimeError(
        "No project selected and current directory is not an nbdev project. "
        "Either cd into an nbdev project, or use set_project(...) to select one."
    )

In [ ]:
#| export
def resolve_selector(selector: Optional[str]) -> Path:
    """Resolve a project selector string to an nbdev project Path.
    
    Selector formats:
    - ``None`` -> returns CURRENT_PROJECT (if set, otherwise raises error)
    - ``'@alias'`` or ``'alias:NAME'`` -> looks up the alias in bookmarks
    - any other string -> treated as a filesystem path
    
    Args:
        selector: Project selector string.
    
    Returns:
        Resolved project root path.
    
    Raises:
        RuntimeError: If selector doesn't resolve to a valid nbdev project.
    """
    if not selector:
        return require_project()
    
    aliases = load_bookmarks()
    sel = selector.strip()
    
    if sel.startswith("alias:"):
        sel = sel.split(":", 1)[1]
    
    if sel.startswith("@"):
        sel = sel[1:]
    
    if sel in aliases:
        p = expand(aliases[sel])
        root = find_project_root(p) or p
        if is_nbdev_project(root):
            return root
        raise RuntimeError(f"Alias '{sel}' points to a directory that is not an nbdev project: {p}")
    
    p = expand(sel)
    root = find_project_root(p) or p
    if is_nbdev_project(root):
        return root
    raise RuntimeError(
        f"Not an nbdev project (requires nbs/ and nbdev settings file: settings.ini/settings.toml/pyproject.toml): {p}"
    )


## Notebook Iteration

### Cache

In [ ]:
#| export
# Cache for notebook listings with time-based invalidation
notebook_cache: Dict[str, Tuple[float, List[Path]]] = {}
NOTEBOOK_CACHE_TTL = 30.0  # Cache TTL in seconds


In [ ]:
#| export
def clear_notebook_cache() -> None:
    """Clear the notebook listing cache.
    
    Removes all cached notebook listings. Call this when notebooks
    are added or removed to force a fresh scan on next access.
    """
    global notebook_cache
    notebook_cache = {}


In [ ]:
#| export
def iter_notebooks_uncached(project: Path) -> List[Path]:
    """Internal: collect notebooks from nbs/ and tutorials/ without caching.
    
    Parameters
    ----------
    project : Path
        Path to the nbdev project root.
    
    Returns
    -------
    List[Path]
        Sorted list of notebook paths, excluding checkpoint directories.
    """
    result: List[Path] = []
    nbs = nbs_dir(project)
    if nbs.exists():
        for p in sorted(nbs.rglob("*.ipynb")):
            if ".ipynb_checkpoints" in str(p):
                continue
            result.append(p)
    tuts = tutorials_dir(project)
    if tuts.exists():
        for p in sorted(tuts.rglob("*.ipynb")):
            if ".ipynb_checkpoints" in str(p):
                continue
            result.append(p)
    return result


In [ ]:
#| export
def iter_notebooks(project: Path, use_cache: bool = True) -> Iterable[Path]:
    """Iterate over all notebooks in the project.
    
    Yields notebooks from nbs/ and tutorials/ directories in sorted order,
    excluding ``.ipynb_checkpoints`` folders.
    
    Results are cached for 30 seconds to improve performance on repeated calls.
    
    Parameters
    ----------
    project : Path
        Path to the nbdev project root.
    use_cache : bool, optional
        Whether to use caching (default True). Set to False to force refresh.
    
    Yields
    ------
    Path
        Paths to notebook files.
    """
    global notebook_cache
    cache_key = str(project)
    
    if use_cache and cache_key in notebook_cache:
        cached_time, cached_list = notebook_cache[cache_key]
        # Check if cache is still valid (TTL)
        if time.time() - cached_time < NOTEBOOK_CACHE_TTL:
            yield from cached_list
            return
    
    # Cache miss or expired - rebuild
    result = iter_notebooks_uncached(project)
    notebook_cache[cache_key] = (time.time(), result)
    yield from result

## Project Summary

In [ ]:
#| export
def project_summary(project: Path) -> Dict[str, Any]:
    """Gather key info about the nbdev project.
    
    Parameters
    ----------
    project : Path
        Path to the nbdev project root.
    
    Returns
    -------
    Dict[str, Any]
        Dictionary with project info: project path, lib_name, nbs_dir,
        has_index_ipynb, has_readme, env_file, settings file, and generation.
    """
    s = settings_dict(project)
    ef = env_file(project)
    settings_file = nbdev_settings_path(project)
    return {
        "project": str(project),
        "lib_name": s.get("lib_name"),
        "nbs_dir": str(nbs_dir(project)),
        "has_index_ipynb": (nbs_dir(project) / "index.ipynb").exists(),
        "has_readme": (project / "README.md").exists(),
        "env_file": str(ef) if ef.exists() else None,
        "nbdev_settings_file": settings_file.name if settings_file is not None else None,
        "nbdev_generation": nbdev_generation(project),
    }


## Notebook I/O

In [ ]:
#| export
def read_nb(path: Path) -> Dict[str, Any]:
    """Read a Jupyter notebook file and return its JSON contents.
    
    Parameters
    ----------
    path : Path
        Path to the notebook file.
    
    Returns
    -------
    Dict[str, Any]
        Parsed notebook JSON data.
    """
    return json.loads(path.read_text(encoding="utf-8"))


In [ ]:
#| export
def write_nb(path: Path, data: Dict[str, Any]) -> None:
    """Write a dict as JSON to a Jupyter notebook file.
    
    Parameters
    ----------
    path : Path
        Path to the notebook file.
    data : Dict[str, Any]
        Notebook data to write.
    """
    path.write_text(json.dumps(data, indent=2), encoding="utf-8")


## Module Resolution

In [ ]:
#| export
def resolve_relative(current: str, rel: str) -> str:
    """Resolve a relative import path to an absolute module path.
    
    Parameters
    ----------
    current : str
        The current module path (e.g., ``'pkg.subpkg.module'``).
    rel : str
        The relative import (e.g., ``'..utils'`` or ``'.helpers'``).
    
    Returns
    -------
    str
        The resolved absolute module path.
    
    Examples
    --------
    >>> resolve_relative('pkg.sub.mod', '.utils')
    'pkg.sub.utils'
    >>> resolve_relative('pkg.sub.mod', '..core')
    'pkg.core'
    """
    m = re.match(RELATIVE_LEVEL_PATTERN, rel)
    if not m:
        return rel
    dots, tail = m.groups()
    level = len(dots)
    parts = current.split(".")
    if "__init__" in parts:
        parts = parts[:-1]
    base_parts = parts[:-level]
    tail_parts = [p for p in tail.split(".") if p]
    return ".".join(base_parts + tail_parts)

In [ ]:
#| export
def abs_module_for_nb(project: Path, nb_path: Path) -> Tuple[str, str]:
    """Determine the library name and absolute module path for a notebook.
    
    Parameters
    ----------
    project : Path
        The nbdev project root.
    nb_path : Path
        Path to the notebook file.
    
    Returns
    -------
    Tuple[str, str]
        A tuple of (lib_name, module_path) where module_path is the dotted
        module name (e.g., ``'subpkg.utils'``).
    """
    s = settings_dict(project)
    lib = s.get("lib_name") or "pkg"
    data = read_nb(nb_path)
    mod = find_default_exp(data)
    nbs_root = nbs_dir(project)
    
    # Check if notebook is in nbs/ directory
    try:
        rel = nb_path.relative_to(nbs_root)
        in_nbs = True
    except ValueError:
        # Notebook is outside nbs/ (e.g., tutorials/)
        # Use project-relative path for module name
        try:
            rel = nb_path.relative_to(project)
        except ValueError:
            rel = Path(nb_path.name)
        in_nbs = False
    
    if not mod:
        parts = list(rel.parts)
        name = Path(parts[-1]).stem
        name = re.sub(DIGIT_PREFIX, "", name)
        if len(parts) > 1:
            mod = ".".join(parts[:-1] + [name])
        else:
            mod = name
    
    # Only check for __init__ notebooks if in nbs/
    if in_nbs and rel.name == "00__init__.ipynb" and len(rel.parts) > 1:
        mod = ".".join(list(rel.parent.parts) + ["__init__"])
    
    return lib, mod

In [ ]:
#| export
def py_to_notebook(project: Path, py_file: Path) -> Optional[Path]:
    """Find the source notebook that generated a Python module file.
    
    Parameters
    ----------
    project : Path
        The nbdev project root directory.
    py_file : Path
        Path to a Python file in the library (e.g., ``lib/utils.py``).
    
    Returns
    -------
    Path or None
        Path to the source notebook, or None if not found.
    
    Notes
    -----
    Searches notebooks by matching the ``default_exp`` directive
    to the Python module path, then falls back to naming convention.
    """
    s = settings_dict(project)
    lib = s.get("lib_name") or "pkg"
    lib_path = project / lib.replace("-", "_")
    
    try:
        rel = py_file.relative_to(lib_path)
    except ValueError:
        return None
    
    parts = list(rel.with_suffix('').parts)
    target_module = '.'.join(parts)
    
    nbs_root = nbs_dir(project)
    for nb_path in iter_notebooks(project):
        data = read_nb(nb_path)
        mod = find_default_exp(data)
        if mod == target_module:
            return nb_path
    
    base_name = py_file.stem
    for nb_path in iter_notebooks(project):
        nb_base = re.sub(NBDEV_FILE_PREFIX, '', nb_path.stem)
        if nb_base == base_name:
            return nb_path
    
    return None


## Module Index

In [ ]:
#| export
import runpy

# Cache for modidx with mtime-based invalidation
_modidx_cache: Dict[str, Tuple[float, Dict[str, Any]]] = {}

def modidx_path(project: Path) -> Path:
    """Return the path to _modidx.py for the project.
    
    Parameters
    ----------
    project : Path
        Path to the nbdev project root.
    
    Returns
    -------
    Path
        Resolved path to ``{lib}/_modidx.py``.
    """
    lib = lib_name(project)
    return (project / lib.replace("-", "_") / "_modidx.py").resolve()


def clear_modidx_cache() -> None:
    """Clear the _modidx.py cache.
    
    Removes all cached modidx data. Call this when _modidx.py
    changes to force a fresh parse on next access.
    """
    global _modidx_cache
    _modidx_cache = {}


def load_modidx(project: Path, use_cache: bool = True) -> Dict[str, Any]:
    """Parse _modidx.py and return the exported dict.
    
    Results are cached based on file modification time for performance.
    
    Parameters
    ----------
    project : Path
        Path to the nbdev project root.
    use_cache : bool, optional
        Whether to use caching (default True). Set to False to force reload.
    
    Returns
    -------
    Dict[str, Any]
        The 'd' dict from _modidx.py containing module/symbol info.
    
    Raises
    ------
    FileNotFoundError
        If _modidx.py doesn't exist.
    ValueError
        If _modidx.py has invalid format (missing 'd' dict).
    """
    global _modidx_cache
    path = modidx_path(project)
    if not path.exists():
        raise FileNotFoundError(f"_modidx.py not found at {path}")
    
    cache_key = str(path)
    current_mtime = path.stat().st_mtime
    
    # Check cache validity (mtime-based)
    if use_cache and cache_key in _modidx_cache:
        cached_mtime, cached_data = _modidx_cache[cache_key]
        if cached_mtime == current_mtime:
            return cached_data
    
    # Cache miss or file changed - reload
    data = runpy.run_path(str(path))
    d = data.get("d") if isinstance(data, dict) else None
    if not isinstance(d, dict):
        raise ValueError("Invalid _modidx.py format: missing dict 'd'")
    
    _modidx_cache[cache_key] = (current_mtime, d)
    return d


def symbol_locations(modidx: Dict[str, Any], symbol: str) -> List[Dict[str, str]]:
    """Find all locations where a symbol is exported.
    
    Parameters
    ----------
    modidx : Dict[str, Any]
        The 'd' dict from _modidx.py.
    symbol : str
        Symbol name to search for.
    
    Returns
    -------
    List[Dict[str, str]]
        List of dicts with 'module' and 'source' keys for each location.
    """
    out: List[Dict[str, str]] = []
    syms = modidx.get("syms", {}) if isinstance(modidx, dict) else {}
    for mod, entries in syms.items():
        if not isinstance(entries, dict):
            continue
        for sym, meta in entries.items():
            if sym.split(".")[-1] != symbol:
                continue
            src = meta.get("source") if isinstance(meta, dict) else None
            out.append({"module": mod, "source": str(src) if src else ""})
    return out


## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

## with_project Decorator

In [ ]:
#| export
from functools import wraps
from typing import Callable

def with_project(fn: Callable[..., Dict[str, Any]]) -> Callable[..., Dict[str, Any]]:
    """Decorator that handles project resolution boilerplate for tool functions.

    Wraps a function that takes a Path as its first argument, converting
    the optional string `project` parameter to a resolved Path. If resolution
    fails, returns ``{'ok': False, 'error': ...}`` instead of raising.

    Parameters
    ----------
    fn : Callable
        Function with signature ``(p: Path, ...) -> Dict[str, Any]``
        where ``p`` is the resolved project path.

    Returns
    -------
    Callable
        Wrapped function with signature ``(project: Optional[str] = None, ...) -> Dict[str, Any]``

    Examples
    --------
    >>> @with_project
    ... def my_tool(p: Path, verbose: bool = False) -> Dict[str, Any]:
    ...     return {'ok': True, 'project': str(p)}
    >>> my_tool()  # Uses current project
    >>> my_tool(project='/path/to/proj')  # Uses specified project
    """
    @wraps(fn)
    def wrapper(project: Optional[str] = None, **kwargs) -> Dict[str, Any]:
        try:
            p = resolve_selector(project)
        except Exception as e:
            return {'ok': False, 'error': str(e)}
        return fn(p, **kwargs)
    return wrapper

In [ ]:
#| hide
# (Moved to cell 22 - notebook caching)

## MCP Configuration Paths

Provider-specific configuration paths for AI tools.

In [ ]:
#| export
def get_vscode_user_dir() -> Path:
    """Get VS Code user settings directory.
    
    Returns platform-specific path:
    - macOS: ~/Library/Application Support/Code/User
    - Windows: %APPDATA%/Code/User
    - Linux: ~/.config/Code/User
    """
    system = platform.system().lower()
    if system == 'darwin':
        return Path.home() / 'Library' / 'Application Support' / 'Code' / 'User'
    elif system == 'windows':
        return Path(os.environ.get('APPDATA', '')) / 'Code' / 'User'
    return Path.home() / '.config' / 'Code' / 'User'

In [ ]:
#| export
def get_vscode_mcp_path() -> Path:
    """Get VS Code MCP configuration path."""
    return get_vscode_user_dir() / 'mcp.json'

In [ ]:
#| export
def get_vscode_settings_path() -> Path:
    """Get VS Code settings.json path."""
    return get_vscode_user_dir() / 'settings.json'

In [ ]:
#| export
def get_claude_config_dir() -> Path:
    """Get Claude Code configuration directory (~/.claude)."""
    return Path.home() / '.claude'

In [ ]:
#| export
def get_claude_config_path() -> Path:
    """Get Claude Code main config path (~/.claude.json)."""
    return Path.home() / '.claude.json'

In [ ]:
#| export
def get_claude_mcp_path() -> Path:
    """Get Claude Code MCP servers config path (~/.claude/mcp_servers.json)."""
    return get_claude_config_dir() / 'mcp_servers.json'

In [ ]:
#| export
def get_codex_config_dir() -> Path:
    """Get Codex configuration directory (~/.codex)."""
    return Path.home() / '.codex'

In [ ]:
#| export
def get_codex_config_path() -> Path:
    """Get Codex config.toml path."""
    return get_codex_config_dir() / 'config.toml'

In [ ]:
#| export
def get_ollama_config_dir() -> Path:
    """Get Ollama configuration directory.
    
    Returns platform-specific path:
    - macOS: ~/.ollama
    - Windows: %LOCALAPPDATA%/Ollama
    - Linux: ~/.ollama
    """
    system = platform.system().lower()
    if system == 'windows':
        return Path(os.environ.get('LOCALAPPDATA', '')) / 'Ollama'
    return Path.home() / '.ollama'

In [ ]:
#| export
def get_ollama_config_path() -> Path:
    """Get Ollama configuration path."""
    return get_ollama_config_dir() / 'config.json'

In [ ]:
#| export
def get_project_mcp_path(project_dir: Optional[Path] = None) -> Path:
    """Get project-level MCP config path (.mcp.json)."""
    project = project_dir or Path.cwd()
    return project / '.mcp.json'

In [ ]:
#| export
def get_project_claude_path(project_dir: Optional[Path] = None) -> Path:
    """Get project-level Claude config path (.claude/mcp.json)."""
    project = project_dir or Path.cwd()
    return project / '.claude' / 'mcp.json'

In [ ]:
#| export
def get_all_config_paths() -> Dict[str, Path]:
    """Get preferred MCP configuration paths for supported providers.

    Returns
    -------
    dict[str, Path]
        Mapping of provider name to selected config path.
    """
    providers = ['vscode', 'cursor', 'claude', 'codex', 'ollama']
    paths: Dict[str, Path] = {}
    for provider in providers:
        selected = select_provider_config_path(provider)
        if selected is not None:
            paths[provider] = selected
    return paths


In [ ]:
#| export
def get_mcp_config_path(provider: str) -> Optional[Path]:
    """Get the preferred config path for a specific MCP provider.

    Parameters
    ----------
    provider : str
        Provider name (vscode, cursor, claude, codex, ollama).

    Returns
    -------
    Optional[Path]
        Selected config path or None if unknown provider.
    """
    return select_provider_config_path(provider)


In [ ]:
#| export
def get_cursor_user_dir() -> Path:
    """Get Cursor user settings directory."""
    system = platform.system().lower()
    if system == 'darwin':
        return Path.home() / 'Library' / 'Application Support' / 'Cursor' / 'User'
    elif system == 'windows':
        return Path(os.environ.get('APPDATA', '')) / 'Cursor' / 'User'
    return Path.home() / '.config' / 'Cursor' / 'User'


def get_cursor_mcp_path() -> Path:
    """Get Cursor MCP configuration path."""
    return get_cursor_user_dir() / 'mcp.json'


def get_cursor_settings_path() -> Path:
    """Get Cursor settings.json path."""
    return get_cursor_user_dir() / 'settings.json'


def get_claude_desktop_config_path() -> Path:
    """Get Claude Desktop config path."""
    system = platform.system().lower()
    if system == 'darwin':
        return Path.home() / 'Library' / 'Application Support' / 'Claude' / 'claude_desktop_config.json'
    elif system == 'windows':
        return Path(os.environ.get('APPDATA', '')) / 'Claude' / 'claude_desktop_config.json'
    return Path.home() / '.config' / 'Claude' / 'claude_desktop_config.json'


def get_codex_json_config_path() -> Path:
    """Get legacy Codex JSON config path (~/.codex/config.json)."""
    return get_codex_config_dir() / 'config.json'


def get_provider_config_paths(provider: str) -> List[Path]:
    """Get all supported config file paths for a provider in priority order."""
    name = provider.lower()
    if name == 'vscode':
        return [get_vscode_mcp_path()]
    if name == 'cursor':
        return [get_cursor_mcp_path()]
    if name == 'claude':
        return [get_claude_config_path(), get_claude_desktop_config_path()]
    if name == 'codex':
        return [get_codex_config_path(), get_codex_json_config_path()]
    if name == 'ollama':
        return [get_ollama_config_path()]
    return []


def select_provider_config_path(provider: str) -> Optional[Path]:
    """Select the active config path for a provider.

    Prefers the first existing file from provider candidates; otherwise
    falls back to the first candidate path.
    """
    candidates = get_provider_config_paths(provider)
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0] if candidates else None
